# Leave-one-HPO-out performance analysis

This notebook audits and summarizes internal-external validation across HPOs. The primary summaries give every held-out HPO equal weight and report medians and interquartile ranges. Patient-pooled performance is intentionally not used as the primary transportability result.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")
sns.set_theme(style="whitegrid", context="notebook")

## Load results

The notebook prefers the revised 2026-10-11 outputs. It falls back to the earlier HPO-level result so the descriptive sections can still run before the revised analysis finishes. Eligibility and patient-level checks require the new files.

In [ ]:
working_dir = Path.cwd()
result_dir = (working_dir if working_dir.name == "3.retrain_regression"
              else working_dir / "3.retrain_regression")

performance_path = result_dir / "finetune_per_hpo_all_lookback_years_leave_one_hpo_out.csv"
if not performance_path.exists():
    raise FileNotFoundError("No leave-one-HPO-out performance file was found.")

eligibility_path = result_dir / "leave_one_hpo_out_eligibility.csv"
predictions_path = result_dir / "patient_level_leave_one_hpo_out_predictions.csv"

performance = pd.read_csv(performance_path)
eligibility = pd.read_csv(eligibility_path) if eligibility_path.exists() else None
predictions = pd.read_csv(predictions_path) if predictions_path.exists() else None

print(f"Performance source: {performance_path.name}")
print(f"Performance rows: {len(performance):,}")
print(f"Eligibility report available: {eligibility is not None}")
print(f"Patient-level predictions available: {predictions is not None}")

## Structural checks

In [ ]:
required_performance_columns = {
    "hpo", "patient_count", "hpo_positive_cases", "lookback_year",
    "aou_roc_auc", "weighted_roc_auc", "ori_roc_auc",
    "aou_pr_auc", "weighted_pr_auc", "ori_pr_auc",
    "aou_brier_score", "weighted_brier_score", "ori_brier_score",
    "aou_ece", "weighted_ece", "ori_ece",
}
missing_columns = required_performance_columns - set(performance.columns)
assert not missing_columns, f"Missing performance columns: {sorted(missing_columns)}"
assert performance["hpo"].notna().all(), "Missing HPO identifiers in performance results."
assert performance["lookback_year"].notna().all(), "Missing lookback years."
assert not performance.duplicated(["hpo", "lookback_year"]).any(), (
    "Expected one performance row per HPO and lookback year."
)

coverage = (performance.groupby("lookback_year")
            .agg(hpo_count=("hpo", "nunique"),
                 patient_count=("patient_count", "sum"),
                 positive_count=("hpo_positive_cases", "sum"))
            .reset_index())
display(coverage)

metric_columns = [column for column in performance.columns
                  if column.endswith(("roc_auc", "pr_auc", "brier_score", "ece"))]
metric_missingness = performance[metric_columns].isna().sum().sort_values(ascending=False)
display(metric_missingness[metric_missingness.gt(0)].rename("missing_values").to_frame())

In [ ]:
if predictions is not None:
    required_prediction_columns = {
        "loho_row_id", "hpo", "lookback_year", "visit_label",
        "aou_probability", "weighted_probability", "original_probability",
    }
    missing = required_prediction_columns - set(predictions.columns)
    assert not missing, f"Missing prediction columns: {sorted(missing)}"
    assert not predictions.duplicated(["lookback_year", "loho_row_id"]).any(), (
        "At least one observation has multiple LOHO predictions."
    )
    probability_columns = ["aou_probability", "weighted_probability", "original_probability"]
    assert predictions[probability_columns].notna().all().all(), "Missing probabilities."
    assert predictions[probability_columns].apply(lambda x: x.between(0, 1).all()).all(), (
        "A probability falls outside [0, 1]."
    )
    expected = performance.groupby("lookback_year")["patient_count"].sum().sort_index()
    observed = predictions.groupby("lookback_year").size().sort_index()
    prediction_coverage = pd.concat([expected.rename("expected"),
                                     observed.rename("observed")], axis=1)
    prediction_coverage["difference"] = (prediction_coverage["observed"] -
                                           prediction_coverage["expected"])
    display(prediction_coverage)
    assert prediction_coverage["difference"].eq(0).all(), "Prediction coverage mismatch."
else:
    warnings.warn("Patient-level prediction file is unavailable; OOF coverage checks were skipped.")

## HPO eligibility and exclusions

In [ ]:
if eligibility is not None:
    eligibility["eligible"] = eligibility["eligible"].astype(bool)
    eligibility_summary = (eligibility.groupby("lookback_year")
                           .agg(total_hpos=("hpo", "nunique"),
                                eligible_hpos=("eligible", "sum"))
                           .reset_index())
    eligibility_summary["excluded_hpos"] = (eligibility_summary["total_hpos"] -
                                                eligibility_summary["eligible_hpos"])
    exclusions = eligibility.loc[~eligibility["eligible"], [
        "lookback_year", "hpo", "patient_count", "positive_count",
        "negative_count", "minority_class_count", "exclusion_reason",
    ]].sort_values(["lookback_year", "hpo"])
    display(eligibility_summary)
    display(exclusions)
else:
    eligibility_summary = None
    exclusions = None
    warnings.warn("Eligibility file is unavailable; exclusions cannot be quantified from the performance file alone.")

## Site-level summaries

Medians and IQRs across held-out HPOs are the primary summaries. This prevents large HPOs from dominating the transportability assessment.

In [ ]:
model_labels = {"aou": "AoU unweighted",
                "weighted": "AoU HPO-weighted",
                "ori": "Original"}
metric_labels = {"roc_auc": "AUROC", "pr_auc": "PR-AUC",
                 "brier_score": "Brier score", "ece": "ECE"}
summary_rows = []
for lookback_year, year_df in performance.groupby("lookback_year"):
    for model, model_label in model_labels.items():
        for metric, metric_label in metric_labels.items():
            values = year_df[f"{model}_{metric}"].dropna()
            summary_rows.append({
                "lookback_year": int(lookback_year),
                "model": model_label,
                "metric": metric_label,
                "hpo_count": len(values),
                "median": values.median(),
                "q1": values.quantile(0.25),
                "q3": values.quantile(0.75),
                "minimum": values.min(),
                "maximum": values.max(),
            })
summary = pd.DataFrame(summary_rows)
summary["median_iqr"] = summary.apply(
    lambda row: f"{row['median']:.3f} ({row['q1']:.3f}–{row['q3']:.3f})", axis=1
)
primary_table = (summary.pivot(index=["lookback_year", "metric"],
                               columns="model", values="median_iqr")
                 .reset_index())
display(primary_table[primary_table['lookback_year'] == 5])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
for ax, metric in zip(axes.flat, ["AUROC", "PR-AUC", "Brier score", "ECE"]):
    plot_df = summary[summary["metric"] == metric]
    for model, model_df in plot_df.groupby("model", sort=False):
        model_df = model_df.sort_values("lookback_year")
        x = model_df["lookback_year"].to_numpy()
        median = model_df["median"].to_numpy()
        q1 = model_df["q1"].to_numpy()
        q3 = model_df["q3"].to_numpy()
        ax.plot(x, median, marker="o", label=model)
        ax.fill_between(x, q1, q3, alpha=0.15)
    ax.set(title=f"Held-out-HPO {metric}", xlabel="Lookback years", ylabel=metric)
    ax.set_xticks(sorted(performance["lookback_year"].unique()))
axes[0, 0].legend(frameon=True)
plt.savefig("performance_metrics_loho_all_lookback_years.png", dpi=300, bbox_inches="tight")
plt.show()

## Between-HPO heterogeneity and paired comparisons

In [ ]:
primary_lookback = 5
primary = performance[performance["lookback_year"] == primary_lookback].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
comparisons = [
    ("ori_roc_auc", "aou_roc_auc", "Original AUROC", "AoU unweighted AUROC"),
    ("aou_roc_auc", "weighted_roc_auc", "AoU unweighted AUROC", "AoU HPO-weighted AUROC"),
]
for ax, (x_col, y_col, x_label, y_label) in zip(axes, comparisons):
    plot_df = primary.dropna(subset=[x_col, y_col])
    lower = min(plot_df[x_col].min(), plot_df[y_col].min())
    upper = max(plot_df[x_col].max(), plot_df[y_col].max())
    padding = max((upper - lower) * 0.05, 0.01)
    limits = (lower - padding, upper + padding)
    sns.scatterplot(data=plot_df, x=x_col, y=y_col, size="patient_count",
                    sizes=(30, 400), alpha=0.65, legend=False, ax=ax)
    ax.plot(limits, limits, color="firebrick", linestyle="--")
    ax.set(xlim=limits, ylim=limits, xlabel=x_label, ylabel=y_label,
           title=f"{primary_lookback}-year lookback")
    ax.set_aspect("equal", adjustable="box")
plt.show()

lowest_sites = primary.nsmallest(10, "aou_roc_auc")[[
    "hpo", "patient_count", "hpo_positive_cases",
    "aou_roc_auc", "weighted_roc_auc", "ori_roc_auc",
]]
display(lowest_sites)

In [ ]:
heatmap_data = performance.pivot(index="hpo", columns="lookback_year", values="aou_roc_auc")
heatmap_data = heatmap_data.assign(site_median=heatmap_data.median(axis=1)).sort_values("site_median")
site_median = heatmap_data.pop("site_median")
plt.figure(figsize=(10, max(8, len(heatmap_data) * 0.24)))
sns.heatmap(heatmap_data, cmap="viridis", vmin=0.5, vmax=1.0,
            cbar_kws={"label": "AUROC"})
plt.title("AoU unweighted AUROC in each held-out HPO")
plt.xlabel("Lookback years")
plt.ylabel("Held-out HPO")
plt.show()

## Sparse-outcome sensitivity analysis

The primary code permits two observations in each outcome class. This sensitivity summary retains HPOs with at least 10 positive and 10 negative outcomes.

In [ ]:
sensitivity = performance.copy()
sensitivity["negative_count"] = (sensitivity["patient_count"] -
                                     sensitivity["hpo_positive_cases"])
sensitivity["minority_class_count"] = sensitivity[[
    "hpo_positive_cases", "negative_count"
]].min(axis=1)
sensitivity["at_least_10_per_class"] = sensitivity["minority_class_count"].ge(10)

sensitivity_rows = []
for year, year_df in sensitivity.groupby("lookback_year"):
    for label, subset in [("Minimum 2 per class", year_df),
                          ("Minimum 10 per class", year_df[year_df["at_least_10_per_class"]])]:
        sensitivity_rows.append({
            "lookback_year": int(year),
            "analysis": label,
            "hpo_count": subset["hpo"].nunique(),
            "aou_auc_median": subset["aou_roc_auc"].median(),
            "aou_auc_q1": subset["aou_roc_auc"].quantile(0.25),
            "aou_auc_q3": subset["aou_roc_auc"].quantile(0.75),
            "weighted_auc_median": subset["weighted_roc_auc"].median(),
        })
sensitivity_summary = pd.DataFrame(sensitivity_rows)
display(sensitivity_summary)

sparse_hpos = sensitivity.loc[~sensitivity["at_least_10_per_class"], [
    "lookback_year", "hpo", "patient_count", "hpo_positive_cases",
    "negative_count", "minority_class_count",
]].sort_values(["lookback_year", "minority_class_count", "hpo"])
display(sparse_hpos)

## DeLong comparisons

These counts are descriptive. Site-wise multiple testing is not adjusted, and conventional DeLong tests condition on the fitted predictions rather than incorporating model-refitting uncertainty.

In [ ]:
comparison_specs = [
    ("AoU unweighted vs original", "delong_p_value_ori_aou", "aou_roc_auc", "ori_roc_auc"),
    ("AoU HPO-weighted vs original", "delong_p_value_ori_weighted", "weighted_roc_auc", "ori_roc_auc"),
    ("AoU HPO-weighted vs unweighted", "delong_p_value_aou_weighted", "weighted_roc_auc", "aou_roc_auc"),
]
delong_rows = []
for year, year_df in performance.groupby("lookback_year"):
    for label, p_column, first_auc, second_auc in comparison_specs:
        estimable = year_df[p_column].notna()
        significant = estimable & year_df[p_column].lt(0.05)
        difference = year_df[first_auc] - year_df[second_auc]
        delong_rows.append({
            "lookback_year": int(year),
            "comparison": label,
            "estimable_hpos": int(estimable.sum()),
            "p_below_0.05": int(significant.sum()),
            "significantly_higher_auc": int((significant & difference.gt(0)).sum()),
            "significantly_lower_auc": int((significant & difference.lt(0)).sum()),
        })
delong_summary = pd.DataFrame(delong_rows)
display(delong_summary)

## Export manuscript tables and draft Results text

In [ ]:
output_dir = result_dir / "analysis_outputs"
output_dir.mkdir(exist_ok=True)
summary.to_csv(output_dir / "loho_site_level_summary.csv", index=False)
primary_table.to_csv(output_dir / "loho_manuscript_table.csv", index=False)
sensitivity_summary.to_csv(output_dir / "loho_sparse_outcome_sensitivity.csv", index=False)
delong_summary.to_csv(output_dir / "loho_delong_descriptive_summary.csv", index=False)
if exclusions is not None:
    exclusions.to_csv(output_dir / "loho_excluded_hpos.csv", index=False)

first_year = summary[(summary["lookback_year"] == 1) & (summary["metric"] == "AUROC")].set_index("model")
last_year = summary[(summary["lookback_year"] == 7) & (summary["metric"] == "AUROC")].set_index("model")
n_hpos = performance.loc[performance["lookback_year"].eq(1), "hpo"].nunique()
n_patients = int(performance.loc[performance["lookback_year"].eq(1), "patient_count"].sum())
results_text = (
    f"Leave-one-HPO-out validation included {n_hpos} evaluable HPOs and "
    f"{n_patients:,} participants. At the 1-year lookback, median held-out-site "
    f"AUROC was {first_year.loc['AoU unweighted', 'median_iqr']} for the unweighted "
    f"AoU model, {first_year.loc['AoU HPO-weighted', 'median_iqr']} for the "
    f"HPO-weighted model, and {first_year.loc['Original', 'median_iqr']} for the "
    f"original model. At the 7-year lookback, corresponding median AUROCs were "
    f"{last_year.loc['AoU unweighted', 'median_iqr']}, "
    f"{last_year.loc['AoU HPO-weighted', 'median_iqr']}, and "
    f"{last_year.loc['Original', 'median_iqr']}, respectively."
)
print(results_text)
print(f"\nTables saved to: {output_dir}")